# Need U AI Lab · Transformers 모델 로드 + 로컬 테스트

**vLLM은 설치하지 않습니다.** 모델을 코랩 GPU에 직접 로드하고, 이미 로드된 모델을 호출하는 작은 FastAPI 연결만 엽니다.
테스트 입력·니쥬 프롬프트·대화·추출·평가·결과 저장은 로컬 AI Lab에서 처리하며, 모델 추론 자체는 코랩 GPU에서 실행됩니다.

Colab Secrets에 `NGROK_AUTHTOKEN`, `AI_LAB_COLAB_KEY`를 등록하고 접근을 허용하세요. 기존 Hugging Face 토큰은 `HF_TOKEN`으로 등록합니다.
로컬 `.env`의 `AI_LAB_COLAB_KEY`는 코랩과 같아야 합니다. 키 값은 코드·출력에 넣지 않습니다.

아래 1 → 2 → 3 셀을 실행합니다. 기본 Qwen3-8B를 유지하며 `MODEL_IDS`에서 모델을 변경할 수 있습니다.
여러 모델을 넣으면 모두 한 GPU에 유지합니다. VRAM에는 가중치와 두 모델의 생성 캐시가 함께 들어가야 합니다. 원본 BF16/FP16 모델 기준이며 AWQ 등 양자화 모델은 별도 로더 설정이 필요합니다.

코랩에 모델을 로드하는 것만으로 로컬 Python에 모델 객체가 생기지는 않습니다. FastAPI + ngrok은 로컬에서 그 모델을 호출하기 위한 연결입니다.


In [ ]:
# 1. 필요한 라이브러리 설치 · 모델 설정 · Secrets 준비 (vLLM 없음)
import os
import subprocess
import sys
import threading
import time

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=5,<6', 'accelerate>=1,<2', 'fastapi>=0.115,<1',
    'uvicorn>=0.32,<1', 'pyngrok==8.1.2', 'jsonschema>=4.23,<5',
], check=True)

from google.colab import userdata

MODEL_IDS = ['Qwen/Qwen3-8B']  # 여기에 원하는 Hugging Face 모델 ID 입력
# 두 모델 예시 (GPU 메모리 확인 후 변경):
# MODEL_IDS = ['kakaocorp/kanana-1.5-8b-instruct-2505', 'Qwen/Qwen3.5-9B']
MAX_CONTEXT_TOKENS = 16384  # 서버가 검사하는 입력 + 최대 출력 토큰 상한
GENERATION_TIMEOUT_SECONDS = 120
TRUST_REMOTE_CODE = False  # 모델 카드에서 요구하는 모델만 True로 변경
PORT = 8000

api_key = userdata.get('AI_LAB_COLAB_KEY').strip()
ngrok_token = userdata.get('NGROK_AUTHTOKEN').strip()
if not api_key or not ngrok_token:
    raise ValueError('AI_LAB_COLAB_KEY와 NGROK_AUTHTOKEN을 설정하세요.')
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
print('설치·Secrets 준비 완료. 키 값은 출력하지 않습니다.')


In [ ]:
# 2. 모델을 코랩 GPU에 직접 로드 (이 셀에는 테스트·평가 코드 없음)
import torch
import transformers
from huggingface_hub import HfApi
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임을 선택하세요.')
if globals().get('api_thread') is not None and api_thread.is_alive():
    raise RuntimeError('연결 API가 실행 중입니다. stop_colab_api() 후 모델을 바꾸세요.')
if globals().get('loaded'):
    raise RuntimeError('모델이 이미 로드되어 있습니다. 재로드하려면 런타임을 다시 시작하세요.')

loaded = {}
gpu_info = f'{torch.cuda.get_device_name(0)} / {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB'
print('실제 GPU:', gpu_info)
for model_id in dict.fromkeys(MODEL_IDS):
    revision = HfApi().model_info(model_id).sha
    config = AutoConfig.from_pretrained(model_id, revision=revision, trust_remote_code=TRUST_REMOTE_CODE)
    if getattr(config, 'quantization_config', None):
        raise ValueError('이 노트북은 원본 BF16/FP16용입니다. 양자화 모델은 별도 로더가 필요합니다.')
    context_config = getattr(config, 'text_config', config)
    native_context = getattr(context_config, 'max_position_embeddings', MAX_CONTEXT_TOKENS)
    if MAX_CONTEXT_TOKENS > native_context:
        raise ValueError(f'{model_id}: 컨텍스트 상한을 {native_context} 이하로 설정하세요.')
    if config.model_type in {'qwen3_5', 'qwen3_5_moe'}:
        model_class = transformers.AutoModelForImageTextToText
    elif config.model_type == 'gemma4_unified':
        model_class = transformers.AutoModelForMultimodalLM
    else:
        model_class = AutoModelForCausalLM
    print('모델 로드 중:', model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=TRUST_REMOTE_CODE)
    model = model_class.from_pretrained(
        model_id, revision=revision, config=config,
        dtype=torch.bfloat16, device_map={'': 0}, trust_remote_code=TRUST_REMOTE_CODE,
    ).eval()
    loaded[model_id] = {
        'model': model, 'tokenizer': tokenizer, 'revision': revision,
        'lock': threading.Lock(),
    }
print('로드 완료:', list(loaded))
print('다음 셀은 이미 로드된 모델에 API 연결만 추가합니다.')


In [ ]:
"""Small authenticated API over models already loaded in a Colab kernel.

This module is embedded in the downloadable notebook. It performs no model
loading, evaluation or result persistence. JSON schemas guide the prompt and
validate the completed output; they do not constrain the generation logits.
"""

import json
import secrets
import time
from typing import Literal
from uuid import uuid4

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse
from jsonschema import Draft202012Validator, SchemaError, ValidationError
from pydantic import BaseModel, Field


class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str = Field(max_length=200000)


class CompletionRequest(BaseModel):
    model: str
    messages: list[Message] = Field(min_length=1, max_length=100)
    max_tokens: int = Field(default=512, ge=1, le=4096)
    temperature: float = Field(default=0.2, ge=0, le=2)
    stream: bool = False
    stream_options: dict | None = None
    response_format: dict | None = None


def prepare_messages(body):
    messages = [message.model_dump() for message in body.messages]
    schema = None
    if body.response_format:
        kind = body.response_format.get("type")
        if kind == "json_schema":
            definition = body.response_format.get("json_schema")
            schema = definition.get("schema") if isinstance(definition, dict) else None
            if not isinstance(schema, dict):
                raise HTTPException(422, "json_schema.schema 객체가 필요합니다.")
        elif kind == "json_object":
            schema = {"type": "object"}
        else:
            raise HTTPException(422, "지원하는 response_format은 json_schema와 json_object입니다.")
        try:
            Draft202012Validator.check_schema(schema)
        except SchemaError:
            raise HTTPException(422, "유효하지 않은 JSON Schema입니다.") from None
        instruction = (
            "\nReturn only valid JSON matching this schema. No markdown or explanation:\n"
            + json.dumps(schema, ensure_ascii=False)
        )
        if messages[0]["role"] == "system":
            messages[0]["content"] += instruction
        else:
            messages.insert(0, {"role": "system", "content": instruction.strip()})
    return messages, schema


def create_model_api(model_ids, api_key, generate):
    """generate(body, messages, started) yields strings and a final metadata dict."""
    if not api_key:
        raise ValueError("모델 API 키가 필요합니다.")
    app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)

    @app.middleware("http")
    async def authenticate(request: Request, call_next):
        supplied = request.headers.get("authorization", "")
        if not secrets.compare_digest(supplied.encode(), f"Bearer {api_key}".encode()):
            return JSONResponse({"error": {"message": "Unauthorized"}}, status_code=401)
        return await call_next(request)

    @app.get("/v1/models")
    def models():
        return {
            "object": "list",
            "data": [{"id": model, "object": "model", "owned_by": "colab"} for model in model_ids],
        }

    @app.post("/v1/chat/completions")
    def completions(body: CompletionRequest):
        started = time.perf_counter()
        if body.model not in model_ids:
            raise HTTPException(404, "로드된 모델 ID를 선택하세요.")
        messages, schema = prepare_messages(body)
        request_id = f"chatcmpl-{uuid4().hex}"
        created = int(time.time())

        def events():
            content = ""
            metadata = None
            for event in generate(body, messages, started):
                if isinstance(event, str):
                    content += event
                    if event:
                        yield event
                else:
                    metadata = event
            if metadata is None or not content.strip():
                raise HTTPException(502, "모델이 완전한 응답을 반환하지 않았습니다.")
            if schema is not None:
                try:
                    Draft202012Validator(schema).validate(json.loads(content))
                except (ValueError, ValidationError):
                    raise HTTPException(
                        422, "생성된 JSON이 요청 스키마와 일치하지 않습니다."
                    ) from None
            yield {**metadata, "content": content}

        def chunk(choices, **extra):
            return {
                "id": request_id,
                "object": "chat.completion.chunk",
                "created": created,
                "model": body.model,
                "choices": choices,
                **extra,
            }

        def sse():
            try:
                for event in events():
                    if isinstance(event, str):
                        data = chunk(
                            [{"index": 0, "delta": {"content": event}, "finish_reason": None}]
                        )
                    else:
                        data = chunk(
                            [{"index": 0, "delta": {}, "finish_reason": event["finish_reason"]}],
                            timings=event.get("timings"),
                        )
                    yield f"data: {json.dumps(data, ensure_ascii=False)}\n\n"
                    if isinstance(event, dict) and (body.stream_options or {}).get("include_usage"):
                        yield f"data: {json.dumps(chunk([], usage=event['usage']))}\n\n"
                yield "data: [DONE]\n\n"
            except Exception as error:
                message = error.detail if isinstance(error, HTTPException) else "모델 생성 실패"
                yield f"data: {json.dumps({'error': {'message': message}}, ensure_ascii=False)}\n\n"

        if body.stream:
            return StreamingResponse(sse(), media_type="text/event-stream")
        try:
            result = None
            for event in events():
                if isinstance(event, dict):
                    result = event
        except HTTPException:
            raise
        except Exception:
            raise HTTPException(502, "모델 생성 실패") from None
        return {
            "id": request_id,
            "object": "chat.completion",
            "created": created,
            "model": body.model,
            "choices": [
                {
                    "index": 0,
                    "message": {"role": "assistant", "content": result["content"]},
                    "finish_reason": result["finish_reason"],
                }
            ],
            "usage": result["usage"],
            "timings": result.get("timings"),
        }

    return app


def transformers_generator(loaded, max_context_tokens=16384, timeout_seconds=120):
    import queue
    import threading

    import torch
    from transformers import StoppingCriteria, StoppingCriteriaList, TextIteratorStreamer

    class StopGeneration(StoppingCriteria):
        def __init__(self, cancelled, deadline):
            self.cancelled = cancelled
            self.deadline = deadline

        def __call__(self, input_ids, scores, **kwargs):
            return self.cancelled.is_set() or time.perf_counter() >= self.deadline

    class TimedStreamer(TextIteratorStreamer):
        def __init__(self, tokenizer, started):
            super().__init__(
                tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=timeout_seconds
            )
            self.started = started
            self.first_token_ms = None

        def put(self, value):
            if not self.next_tokens_are_prompt and self.first_token_ms is None:
                self.first_token_ms = (time.perf_counter() - self.started) * 1000
            super().put(value)

    def generate(body, messages, started):
        entry = loaded[body.model]
        tokenizer, model = entry["tokenizer"], entry["model"]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        prompt_tokens = inputs["input_ids"].shape[-1]
        if prompt_tokens + body.max_tokens > max_context_tokens:
            raise HTTPException(422, "입력과 최대 출력 토큰이 설정한 컨텍스트 상한을 초과했습니다.")
        if not entry["lock"].acquire(timeout=timeout_seconds):
            raise HTTPException(429, "이 모델의 이전 요청이 아직 실행 중입니다.")
        cancelled = threading.Event()
        deadline = time.perf_counter() + timeout_seconds
        streamer = TimedStreamer(tokenizer, started)
        result = {}
        options = dict(
            **inputs,
            max_new_tokens=body.max_tokens,
            do_sample=body.temperature > 0,
            streamer=streamer,
            stopping_criteria=StoppingCriteriaList([StopGeneration(cancelled, deadline)]),
            pad_token_id=tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id,
        )
        if body.temperature > 0:
            options["temperature"] = body.temperature

        def run():
            try:
                with torch.inference_mode():
                    output = model.generate(**options)
                result["tokens"] = output.shape[-1] - prompt_tokens
                result["timed_out"] = time.perf_counter() >= deadline
            except Exception:
                result["failed"] = True
                streamer.end()
            finally:
                entry["lock"].release()

        thread = threading.Thread(target=run, daemon=True)
        try:
            thread.start()
        except BaseException:
            entry["lock"].release()
            raise
        try:
            yield from streamer
            thread.join(timeout=timeout_seconds)
            if thread.is_alive() or result.get("timed_out"):
                raise HTTPException(504, "모델 생성 시간 상한을 초과했습니다.")
            if result.get("failed"):
                raise HTTPException(502, "모델 생성 실패")
            count = result["tokens"]
            yield {
                "finish_reason": "length" if count >= body.max_tokens else "stop",
                "usage": {
                    "prompt_tokens": prompt_tokens,
                    "completion_tokens": count,
                    "total_tokens": prompt_tokens + count,
                },
                "timings": {"server_first_token_ms": streamer.first_token_ms},
            }
        except queue.Empty:
            raise HTTPException(504, "모델 스트림 시간 상한을 초과했습니다.") from None
        finally:
            cancelled.set()

    return generate



# 3. 이미 로드된 모델을 로컬 AI Lab에서 호출하도록 API 연결
import socket
import uvicorn
from pyngrok import ngrok

if globals().get('api_thread') is not None and api_thread.is_alive():
    raise RuntimeError('API가 이미 실행 중입니다. 기존 URL을 사용하세요.')
with socket.socket() as check:
    if check.connect_ex(('127.0.0.1', PORT)) == 0:
        raise RuntimeError('API 포트가 사용 중입니다. 기존 서버를 종료하세요.')

app = create_model_api(
    list(loaded), api_key,
    transformers_generator(loaded, MAX_CONTEXT_TOKENS, GENERATION_TIMEOUT_SECONDS),
)
api_server = uvicorn.Server(uvicorn.Config(
    app, host='127.0.0.1', port=PORT, log_level='warning', access_log=False,
))
api_thread = threading.Thread(target=api_server.run, daemon=True)
api_thread.start()
tunnel = None

def stop_colab_api():
    global tunnel
    try:
        if tunnel is not None:
            ngrok.disconnect(tunnel.public_url)
            tunnel = None
    finally:
        api_server.should_exit = True
        api_thread.join(timeout=10)
    print('API 연결 종료. 모델은 GPU에 남습니다. 자원 반환은 런타임 삭제로 진행하세요.')

try:
    for _ in range(100):
        if api_server.started:
            break
        if not api_thread.is_alive():
            raise RuntimeError('API 시작 실패')
        time.sleep(0.1)
    else:
        raise TimeoutError('API 시작 시간 초과')
    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(f'http://127.0.0.1:{PORT}', bind_tls=True, inspect=False)
    print('API Base URL:', tunnel.public_url + '/v1')
    for model_id, entry in loaded.items():
        print(json.dumps({
            'name': 'Colab · ' + model_id.split('/')[-1],
            'provider': 'openai', 'environment': 'colab',
            'base_url': tunnel.public_url + '/v1', 'model': model_id,
            'revision': entry['revision'], 'precision': 'BF16',
            'hardware': f'{gpu_info} · Transformers {transformers.__version__}'[:200],
            'api_key_env': 'AI_LAB_COLAB_KEY',
        }, ensure_ascii=False, indent=2))
except BaseException:
    stop_colab_api()
    raise


## 로컬 AI Lab 사용

API Base URL은 3번 셀에서 출력한 `https://.../v1`입니다. Colab 노트북 주소나 API 키를 넣지 않습니다.
AI Lab → 모델 연결 → Colab GPU 연결에 출력 정보를 저장하고 연결 검사 후 응답·추출 역할에 선택하세요.
여러 모델을 로드했다면 같은 URL로 연결을 각각 등록하고 모델 ID만 구분합니다. 관리 API와 워커는 로컬에서 실행합니다.

- 일반 응답·SSE 스트리밍·토큰 usage를 제공합니다.
- JSON Schema는 프롬프트에 넣고 생성 완료 후 검증합니다. 실패를 오류로 반환하며, 토큰 생성 자체를 스키마로 강제하지 않습니다. 니쥬 실제 스키마로 품질을 확인하세요.
- 모델별 요청은 한 번에 하나씩 처리합니다. 서로 다른 모델의 요청은 각 스레드에서 같은 GPU를 공유합니다. 최적화된 서빙 런타임의 처리량을 재현하지 않습니다.
- API 응답 `timings.server_first_token_ms`는 서버가 첫 생성 토큰을 받기까지의 시간입니다. 스트리밍 텍스트는 단어 단위로 모일 수 있으므로 AI Lab의 첫 내용 청크 시간과 구분합니다. 현재 AI Lab 서비스 경로는 이 서버 측 timing을 저장하지 않습니다.
- Transformers의 무제약 생성·메모리 할당·처리 방식 기준으로 품질과 속도를 측정합니다. 실제 배포 런타임에서는 다시 측정해야 합니다.
- 연결 API는 모델을 새로 로드하지 않습니다. `stop_colab_api()`는 연결만 종료합니다. GPU 모델과 코랩 자원을 반환하려면 연결 해제 및 런타임 삭제를 선택합니다.
- `trust_remote_code`가 필요한 모델은 모델 카드를 확인하고 설정을 변경합니다. 실제 commit을 고정해 로드합니다.

[Transformers 채팅 템플릿](https://huggingface.co/docs/transformers/en/chat_templating) · [스트리밍 안내](https://huggingface.co/docs/transformers/en/internal/generation_utils#streamers) · [ngrok Colab 연결](https://ngrok.com/docs/using-ngrok-with/googleColab)


In [ ]:
# 종료할 때만 주석을 해제해 실행하세요.
# stop_colab_api()
# 모델과 GPU 자원을 반환하려면 Colab의 연결 해제 및 런타임 삭제를 선택하세요.
